# FCB 1280 v4: 오라클 주입 상한 + 출력 증류 스크리닝 (K1–K3)

- 탐지기: YOLO26s (Ultralytics 8.4.165) 고정, 입력 1280×720 → letterbox 1280×736
- Input: `uvg_build_dataset`(uvg_cctv), `uvg_eval3_gop60`(cctv_gop60, JobFair 포함), `uvg_B_data_prep`(bdata)
- Accelerator: GPU T4 x2, Internet on, **대화형 세션에서 위에서부터 순서대로 실행**
- 결과: `/kaggle/working/fcb_runs/` → 마지막 셀에서 `fcb_runs.zip`으로 묶음

## v3 대비 변경
- `fcb_lib.detect_from_feats` 추가 (`detect`가 이걸 호출, 결과 동일)
- `fcb_lib.kd_loss`, `anchor_small_weight` 추가
- `fcb_train.oracle_eval` 추가, 캐시에 원본 P5도 저장
- 새 설정 키: `lam_kd`, `kd_small`, `lam_reg` (기본 0 / False / 0 → 기존 run과 동일)


In [ ]:
!pip install -q ultralytics==8.4.165

In [ ]:
%%writefile /kaggle/working/fcb_lib.py
"""FCB(주파수 기반 경량 Feature Correction Branch) 1280 공통 라이브러리
YOLO26s (Ultralytics 8.4.165) 고정, 입력 1280x720 -> letterbox 1280x736 (위아래 8px, 114)
"""
import json, subprocess, contextlib, io, time
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F

W, H, PAD = 1280, 720, 8
HP = H + 2 * PAD                      # 736
GH, GW = HP // 8, W // 8              # DCT 격자 92 x 160 (= P3 격자)
CCTV = [1, 2, 3, 4, 6, 8]             # person, bicycle, car, motorcycle, bus, truck (COCO 80 인덱스 + 1)
BALL = [33]
SMALL = 32 ** 2

# ---------------- 디코딩 ----------------
def read_yuv(video, n=None):
    """yuv420p 디코딩 -> (Y[N,720,1280], U[N,360,640], V[N,360,640]) uint8"""
    p = subprocess.Popen(["ffmpeg", "-v", "error", "-i", video, "-f", "rawvideo", "-pix_fmt", "yuv420p", "-"],
                         stdout=subprocess.PIPE, stderr=subprocess.DEVNULL if n else None)
    fb = W * H * 3 // 2; Y, U, V = [], [], []
    while n is None or len(Y) < n:
        b = p.stdout.read(fb)
        if len(b) < fb: break
        a = np.frombuffer(b, np.uint8)
        Y.append(a[:W * H].reshape(H, W)); U.append(a[W * H:W * H * 5 // 4].reshape(H // 2, W // 2))
        V.append(a[W * H * 5 // 4:].reshape(H // 2, W // 2))
    p.stdout.close(); p.wait()
    return np.stack(Y), np.stack(U), np.stack(V)

def yuv_to_bgr(y, u, v):
    """OpenCV I420 -> BGR (사전실험 C·D와 같은 변환)"""
    return cv2.cvtColor(np.concatenate([y.ravel(), u.ravel(), v.ravel()]).reshape(H * 3 // 2, W), cv2.COLOR_YUV2BGR_I420)

# ---------------- GT (사전실험 C·D build_gt와 같은 규칙: 30fps 프레임 i = 원본 라벨 키 2i+1) ----------------
def build_gt(raw, n):
    """프레임별 [cat_id(1~80), x1, y1, x2, y2] (720p 좌표) 배열 리스트"""
    out = []
    for i in range(n):
        rows = []
        for o in raw.get(str(2 * i + 1), []):
            x1, y1, x2, y2 = o["x_min"] * W, o["y_min"] * H, o["x_max"] * W, o["y_max"] * H
            if x2 - x1 > 0 and y2 - y1 > 0: rows.append([int(o["class_id"]), x1, y1, x2, y2])
        out.append(np.array(rows, np.float32).reshape(-1, 5))
    return out

def coco_gt(gts, track=None):
    images, anns, aid = [], [], 0
    for i, g in enumerate(gts):
        images.append({"id": i, "width": W, "height": H})
        for c, x1, y1, x2, y2 in g:
            aid += 1; w, h = x2 - x1, y2 - y1
            anns.append({"id": aid, "image_id": i, "category_id": int(c), "bbox": [float(x1), float(y1), float(w), float(h)],
                         "area": float(w * h), "iscrowd": 0})
    return {"images": images, "annotations": anns, "categories": [{"id": k} for k in range(1, 81)]}

def coco_ap(gt, preds, classes, img_ids=None):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    if not preds: return None
    with contextlib.redirect_stdout(io.StringIO()):
        cg = COCO(); cg.dataset = gt; cg.createIndex(); cd = cg.loadRes(preds)
        E = COCOeval(cg, cd, "bbox"); E.params.catIds = classes
        if img_ids is not None: E.params.imgIds = list(img_ids)
        E.evaluate(); E.accumulate(); E.summarize()
    f = lambda x: None if x < 0 else round(100 * x, 2)
    s = E.stats
    return {"AP": f(s[0]), "AP50": f(s[1]), "APs": f(s[3]), "APm": f(s[4]), "APl": f(s[5]), "ARs": f(s[9])}

# ---------------- 입력 텐서 ----------------
def frame_tensors(y, u, v, flip=False):
    """한 프레임 -> (rgb[3,736,1280] float 0~1, y[1,736,1280] float 0~255). Ultralytics LetterBox(114)·BGR->RGB와 동일"""
    if flip: y, u, v = y[:, ::-1], u[:, ::-1], v[:, ::-1]
    y, u, v = np.ascontiguousarray(y), np.ascontiguousarray(u), np.ascontiguousarray(v)
    bgr = cv2.copyMakeBorder(yuv_to_bgr(y, u, v), PAD, PAD, 0, 0, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    rgb = torch.from_numpy(np.ascontiguousarray(bgr[..., ::-1].transpose(2, 0, 1))).float() / 255.0
    yp = torch.from_numpy(cv2.copyMakeBorder(y, PAD, PAD, 0, 0, cv2.BORDER_CONSTANT, value=114)).float()[None]
    return rgb, yp

def flip_boxes(g):
    g = g.copy(); x1 = W - g[:, 3]; g[:, 3] = W - g[:, 1]; g[:, 1] = x1; return g

def yolo_targets(gts):
    """프레임별 GT -> Ultralytics loss batch dict (1280x736 기준 정규화 xywh, cls 0~79)"""
    bi, cl, bb = [], [], []
    for j, g in enumerate(gts):
        if len(g) == 0: continue
        x1, y1, x2, y2 = g[:, 1], g[:, 2] + PAD, g[:, 3], g[:, 4] + PAD
        bi.append(np.full(len(g), j)); cl.append(g[:, 0] - 1)
        bb.append(np.stack([(x1 + x2) / 2 / W, (y1 + y2) / 2 / HP, (x2 - x1) / W, (y2 - y1) / HP], 1))
    if not bi: return {"batch_idx": torch.zeros(0), "cls": torch.zeros(0, 1), "bboxes": torch.zeros(0, 4)}
    return {"batch_idx": torch.from_numpy(np.concatenate(bi)).float(),
            "cls": torch.from_numpy(np.concatenate(cl)).float().view(-1, 1),
            "bboxes": torch.from_numpy(np.concatenate(bb)).float()}

def small_masks(gts, shapes):
    """작은 객체(원해상도 면적 < 32²) 박스와 겹치는 셀 = 1. shapes: [(h,w) per level], stride = HP/h"""
    out = []
    for (h, w) in shapes:
        s = HP / h; m = torch.zeros(len(gts), 1, h, w)
        for j, g in enumerate(gts):
            for c, x1, y1, x2, y2 in g:
                if (x2 - x1) * (y2 - y1) >= SMALL: continue
                c0, c1 = int(x1 // s), int(np.ceil(x2 / s)); r0, r1 = int((y1 + PAD) // s), int(np.ceil((y2 + PAD) / s))
                m[j, 0, r0:max(r1, r0 + 1), c0:max(c1, c0 + 1)] = 1
        out.append(m)
    return out

# ---------------- 8x8 블록 DCT (정규직교 DCT-II, scipy dctn norm='ortho'와 동일) ----------------
def dct_basis():
    x = np.arange(8); c = np.cos((2 * x[None] + 1) * x[:, None] * np.pi / 16) * np.sqrt(2 / 8); c[0] = np.sqrt(1 / 8)
    b = np.einsum("ux,vy->uvxy", c, c).reshape(64, 1, 8, 8)       # 채널 k = u*8+v (u: 세로 주파수, v: 가로)
    return torch.from_numpy(b).float()
UV = (np.arange(8)[:, None] + np.arange(8)[None]).ravel()
BANDS = {"DC": np.where(UV == 0)[0], "Low": np.where((UV >= 1) & (UV <= 2))[0],
         "Mid": np.where((UV >= 3) & (UV <= 6))[0], "High": np.where(UV >= 7)[0], "All": np.arange(64)}

class BlockDCT(nn.Module):
    def __init__(self):
        super().__init__(); self.register_buffer("basis", dct_basis())
    def forward(self, y):                    # (B,1,H,W) 0~255 -> (B,64,H/8,W/8)
        return F.conv2d(y, self.basis, stride=8)

def dct_np(y):                               # (H,W) -> (H/8, W/8, 64), 통계 계산용
    h, w = y.shape
    from scipy.fft import dctn
    c = dctn(y.astype(np.float64).reshape(h // 8, 8, w // 8, 8).transpose(0, 2, 1, 3), axes=(2, 3), norm="ortho")
    return c.reshape(h // 8, w // 8, 64)

# ---------------- FCB ----------------
class DWBlock(nn.Module):
    def __init__(self, c, stride=1, res=True):
        super().__init__(); self.res = res and stride == 1
        self.dw = nn.Conv2d(c, c, 3, stride, 1, groups=c); self.pw = nn.Conv2d(c, c, 1)
        self.gn = nn.GroupNorm(8, c); self.act = nn.SiLU()
    def forward(self, x):
        y = self.act(self.gn(self.pw(self.dw(x)))); return x + y if self.res else y

class FCB(nn.Module):
    """Y -> DCT -> 정규화 -> Stem -> Block x2 (F8) -> H3 (Δ3) / Down (F16) -> H4 (Δ4), Aux(F8 -> 목표 대역 ΔD)"""
    def __init__(self, mu, sigma, in_band="All", tgt_band="Mid", C=64, aux=True):
        super().__init__()
        self.in_idx = torch.as_tensor(BANDS[in_band]); self.tgt_idx = torch.as_tensor(BANDS[tgt_band])
        self.dct = BlockDCT()
        self.register_buffer("mu", torch.as_tensor(mu, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
        self.register_buffer("sigma", torch.as_tensor(sigma, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
        cin = len(self.in_idx)
        self.stem = nn.Sequential(nn.Conv2d(cin, C, 1), nn.GroupNorm(8, C), nn.SiLU())
        self.blocks = nn.Sequential(DWBlock(C), DWBlock(C))
        self.h3 = nn.Conv2d(C, 128, 1); self.down = DWBlock(C, stride=2); self.h4 = nn.Conv2d(C, 256, 1)
        for h in (self.h3, self.h4): nn.init.zeros_(h.weight); nn.init.zeros_(h.bias)
        self.aux = nn.Conv2d(C, len(self.tgt_idx), 1) if aux else None
    def forward(self, y):
        d = self.dct(y)[:, self.in_idx]
        f8 = self.blocks(self.stem((d - self.mu) / self.sigma))
        d3 = self.h3(f8); d4 = self.h4(self.down(f8))
        return d3, d4, (self.aux(f8) if (self.aux is not None and self.training) else None)
    def n_params(self, infer=True):
        return sum(p.numel() for n, p in self.named_parameters() if not (infer and n.startswith("aux")))

class CondFCB(nn.Module):
    """보정 모듈 변형 (탐지기 고정, Detect 직전 주입, 출력 0 초기화)
    arch="feat": 압축 P3·P4 feature만 입력 (DCT 없음)
    arch="cond": DCT(Y) 경로 + 압축 P3·P4 feature 합산 (설계 문서 FCB-cond)"""
    needs_feat = True
    def __init__(self, mu, sigma, arch="cond", in_band="All", tgt_band="Mid", C=64, aux=True):
        super().__init__(); self.arch = arch
        self.in_idx = torch.as_tensor(BANDS[in_band]); self.tgt_idx = torch.as_tensor(BANDS[tgt_band])
        gn = lambda: nn.GroupNorm(8, C)
        self.s3 = nn.Sequential(nn.Conv2d(128, C, 1), gn(), nn.SiLU()); self.s4 = nn.Sequential(nn.Conv2d(256, C, 1), gn(), nn.SiLU())
        if arch == "cond":
            self.dct = BlockDCT()
            self.register_buffer("mu", torch.as_tensor(mu, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
            self.register_buffer("sigma", torch.as_tensor(sigma, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
            self.stem = nn.Sequential(nn.Conv2d(len(self.in_idx), C, 1), gn(), nn.SiLU())
            self.down = DWBlock(C, stride=2)
        self.b8 = nn.Sequential(DWBlock(C), DWBlock(C)); self.b16 = nn.Sequential(DWBlock(C), DWBlock(C))
        self.h3 = nn.Conv2d(C, 128, 1); self.h4 = nn.Conv2d(C, 256, 1)
        for h in (self.h3, self.h4): nn.init.zeros_(h.weight); nn.init.zeros_(h.bias)
        self.aux = nn.Conv2d(C, len(self.tgt_idx), 1) if aux else None
    def forward(self, y, f3, f4):
        x8 = self.s3(f3.float())
        if self.arch == "cond": x8 = x8 + self.stem((self.dct(y)[:, self.in_idx] - self.mu) / self.sigma)
        f8 = self.b8(x8)
        x16 = self.s4(f4.float()) + (self.down(f8) if self.arch == "cond" else 0)
        f16 = self.b16(x16)
        return self.h3(f8), self.h4(f16), (self.aux(f8) if (self.aux is not None and self.training) else None)
    def n_params(self, infer=True):
        return sum(p.numel() for n, p in self.named_parameters() if not (infer and n.startswith("aux")))

def make_fcb(cfg, mu, sg):
    if cfg.get("arch", "fcb") == "fcb": return FCB(mu, sg, cfg["in_band"], cfg["tgt_band"], cfg["C"])
    return CondFCB(mu, sg, cfg["arch"], cfg["in_band"], cfg["tgt_band"], cfg["C"])

def run_fcb(fcb, y, f):
    return fcb(y, f[0], f[1]) if getattr(fcb, "needs_feat", False) else fcb(y)

# ---------------- 탐지기 ----------------
def load_detector(weights="yolo26s.pt", device="cuda"):
    from ultralytics import YOLO
    from ultralytics.cfg import get_cfg
    m = YOLO(weights).model.float().to(device)
    m.args = get_cfg()                                # loss gain(box 7.5, cls 0.5, dfl 1.5) 기본값
    for p in m.parameters(): p.requires_grad_(False)
    return m

def set_train_mode(m):
    """Detect 학습 출력(dict)을 쓰기 위해 train(), BatchNorm은 eval 고정"""
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm): mod.eval()

def neck_feats(m, x):
    """Detect 직전까지 실행 -> [P3, P4, P5]"""
    y = []
    for mod in m.model[:-1]:
        if mod.f != -1: x = y[mod.f] if isinstance(mod.f, int) else [x if j == -1 else y[j] for j in mod.f]
        x = mod(x); y.append(x if mod.i in m.save else None)
    return [y[j] for j in m.model[-1].f]

def det_loss_fn(m):
    """one2many 손실만 (E2ELoss의 o2m/o2o 가중·감쇠 미사용)"""
    from ultralytics.utils.loss import v8DetectionLoss
    return v8DetectionLoss(m, tal_topk=10)

def l_det(crit, preds, batch):
    loss, items = crit(preds["one2many"], batch)
    return loss.sum() / preds["one2many"]["boxes"].shape[0], items

def rel_err(p, r, mask=None):
    """샘플별 ||p-r||² / ||r||², mask 있으면 마스크 영역만. 마스크 합 0인 샘플 제외. 반환 (평균, 유효 샘플 수)"""
    if mask is not None:
        mask = F.interpolate(mask, size=p.shape[-2:], mode="nearest") if mask.shape[-2:] != p.shape[-2:] else mask
        p, r = p * mask, r * mask
    num = ((p - r) ** 2).flatten(1).sum(1); den = (r ** 2).flatten(1).sum(1)
    ok = den > 0
    if ok.sum() == 0: return p.new_zeros(()), 0
    return (num[ok] / den[ok]).mean(), int(ok.sum())

# ---------------- 출력 증류 (원본 프레임 탐지기 출력 = 교사) ----------------
def kd_loss(s_out, t_out, anchor_w=None):
    """s_out/t_out: one2many dict (scores B×80×N 로짓, boxes B×4×N). 교사 최대 클래스 확률로 앵커 가중"""
    ts = torch.sigmoid(t_out["scores"].float()); ss = s_out["scores"].float()
    wt = ts.max(1).values                                                   # B×N 교사 확신도 (박스 가중)
    wc = torch.maximum(wt, torch.sigmoid(ss.detach()).max(1).values)          # 분류: 학생 오탐 앵커도 포함
    if anchor_w is not None: wt, wc = wt * anchor_w, wc * anchor_w
    bce = F.binary_cross_entropy_with_logits(ss, ts, reduction="none").sum(1)
    l1 = (s_out["boxes"].float() - t_out["boxes"].float()).abs().sum(1)
    return (wc * bce).sum() / (wc.sum() + 1.0), (wt * l1).sum() / (wt.sum() + 1.0)

def anchor_small_weight(gts, classes=None, boost=5.0):
    """작은 GT 박스(원해상도 면적 < 32²) 안 앵커에 boost배 가중. P3·P4·P5 순서로 펼침 (P5는 1)"""
    gg = [g if classes is None else g[np.isin(g[:, 0], classes)] for g in gts]
    ms = small_masks(gg, [(92, 160), (46, 80)])
    w = [1 + (boost - 1) * mm.flatten(1) for mm in ms] + [torch.ones(len(gts), 23 * 40)]
    return torch.cat(w, 1)

# ---------------- 추론 (Ultralytics predict와 같은 후처리) ----------------
@torch.no_grad()
def detect(m, rgb, fcb=None, y=None, conf=0.001, iou=0.7, max_det=300):
    """rgb (B,3,736,1280) -> 프레임별 [x,y,w,h,score,cat(1~80)] (720p 좌표). fcb 있으면 P3·P4에 Δ 더함"""
    from ultralytics.utils.nms import non_max_suppression
    m.eval()
    f = neck_feats(m, rgb)
    if fcb is not None:
        fcb.eval(); d3, d4, _ = run_fcb(fcb, y, f); f = [f[0] + d3, f[1] + d4, f[2]]
    return detect_from_feats(m, f, conf, iou, max_det)

@torch.no_grad()
def detect_from_feats(m, f, conf=0.001, iou=0.7, max_det=300):
    """[P3, P4, P5] -> Detect(eval) -> NMS -> 프레임별 [x,y,w,h,score,cat] (720p)"""
    from ultralytics.utils.nms import non_max_suppression
    m.eval(); out = m.model[-1](f)
    pr = non_max_suppression(out[0] if isinstance(out, tuple) else out, conf, iou, None, False, max_det=max_det, nc=0,
                             max_time_img=10.0)   # 기본 0.05(배치 8 -> 2.4초)면 conf 0.001에서 시간 초과로 검출이 잘림
    res = []
    for p in pr:
        p = p.cpu().numpy(); b = p[:, :4].copy(); b[:, [1, 3]] -= PAD
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
        res.append(np.column_stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1], p[:, 4], p[:, 5] + 1]))
    return res

def to_coco(dets, ids):
    return [{"image_id": int(i), "category_id": int(r[5]), "bbox": [float(v) for v in r[:4]], "score": float(r[4])}
            for i, d in zip(ids, dets) for r in d]


In [ ]:
%%writefile /kaggle/working/fcb_train.py
# FCB 1280 ② 학습·평가: RUNS의 각 설정을 학습 -> 검증셋 평가 -> (정식 run이면) 평가셋 평가, 결과 CSV·가중치 저장
# 사용: RUNS = [dict(name=..., ...)] 정의 후 %run fcb_train.py  (기본값은 DEFAULT 참고)
import os, sys, glob, json, time, math, gc
import numpy as np, pandas as pd, torch
from concurrent.futures import ThreadPoolExecutor
POOL = ThreadPoolExecutor(4)   # cv2 색변환·패딩 병렬 (GIL 해제)
sys.path.insert(0, "/kaggle/working"); import fcb_lib as L
OUT = "/kaggle/working/fcb_runs"; os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"; AMP = DEV == "cuda"
TRAIN_ALL = ["HighwayDrive", "WinterDrive", "TrafficLights", "OfficeWalk", "SpotRobot", "BoatReverse"]
EVAL = {"HighwayView": "cctv", "CampusView": "cctv", "VolleyballGame": "ball", "FloorballGame": "ball", "FloorballTrain": "ball"}
DEFAULT = dict(name="fcb", in_band="All", tgt_band="Mid", C=64, val_seq="WinterDrive",
               train_kbps=[2000, 1000, 500, 250], frame_step=1, epochs=20, batch=8, lr=1e-3, wd=1e-4, warmup=1,
               lam_det=1.0, lam_feat=1.0, lam_freq=1.0, flip=True, cache=False, seed=0,
               eval_val_kbps=[500, 250], eval_full=False, eval_kbps=["ref", 4000, 1000, 500, 250], ckpt=None, arch="fcb",
               lam_kd=0.0, kd_small=False, lam_reg=0.0)
SCREEN = dict(cache=True, flip=False, frame_step=2, train_kbps=[500, 250], epochs=10, eval_full=True, eval_kbps=[500, 250])
CACHE = {}   # 스크리닝 run끼리 같은 (학습 시퀀스, frame_step, 비트레이트)면 탐지기 feature 캐시 재사용
RUNS = globals().get("RUNS", [dict(name="smoke", epochs=1, frame_step=10)])
STAMP = time.strftime("%m%d_%H%M")

def find_dir(name):
    c = glob.glob(f"/kaggle/input/**/{name}", recursive=True); return c[0] if c else None
REF, GOP60, BDATA = find_dir("uvg_cctv"), find_dir("cctv_gop60"), find_dir("bdata")
def comp_path(seq, k):
    for root in (BDATA, GOP60):
        if root and os.path.exists(f"{root}/{seq}/cctv_{k}k.mp4"): return f"{root}/{seq}/cctv_{k}k.mp4"
    raise FileNotFoundError(f"{seq} {k}k")
def n_frames(s): return json.load(open(f"{REF}/{s}/done.json"))["ref_frames"]
def load_seq(s, kbs, ref=True):
    n = n_frames(s); d = {"comp": {k: L.read_yuv(comp_path(s, k), n) for k in kbs},
                         "gt": L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n)}
    if ref: d["ref"] = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n)
    return d

m = L.load_detector("yolo26s.pt", DEV); crit = L.det_loss_fn(m); dct = L.BlockDCT().to(DEV)
DATA = {}
def ensure(seqs, kbs):
    for s in seqs:
        if s not in DATA: DATA[s] = load_seq(s, kbs)
        for k in kbs:
            if k not in DATA[s]["comp"]: DATA[s]["comp"][k] = L.read_yuv(comp_path(s, k), n_frames(s))

def dct_stats(seqs, kbs):
    a = np.zeros((5, 64))
    with torch.no_grad():
        for s in seqs:
            R = DATA[s]["ref"][0]
            for k in kbs:
                C = DATA[s]["comp"][k][0]
                for i in range(0, len(R), 32):
                    yr = dct(torch.from_numpy(R[i:i + 32]).to(DEV).float()[:, None]).double()
                    yc = dct(torch.from_numpy(C[i:i + 32]).to(DEV).float()[:, None]).double(); dd = yr - yc
                    a[0] += yc.sum((0, 2, 3)).cpu().numpy(); a[1] += (yc ** 2).sum((0, 2, 3)).cpu().numpy()
                    a[2] += dd.sum((0, 2, 3)).cpu().numpy(); a[3] += (dd ** 2).sum((0, 2, 3)).cpu().numpy()
                    a[4] += yc.shape[0] * yc.shape[2] * yc.shape[3]
    n = a[4]; mu = a[0] / n; sg = np.sqrt(a[1] / n - mu ** 2); mud = a[2] / n; sgd = np.sqrt(a[3] / n - mud ** 2)
    return mu, sg, sgd

def make_batch(items, flip):
    fls = [flip and np.random.rand() < 0.5 for _ in items]
    jobs = [POOL.submit(L.frame_tensors, *(x[i] for x in DATA[s]["comp"][k]), flip=fl) for (s, i, k), fl in zip(items, fls)]
    jobs += [POOL.submit(L.frame_tensors, *(x[i] for x in DATA[s]["ref"]), flip=fl) for (s, i, k), fl in zip(items, fls)]
    res = [j.result() for j in jobs]; n = len(items)
    gts = [L.flip_boxes(DATA[s]["gt"][i]) if fl else DATA[s]["gt"][i] for (s, i, k), fl in zip(items, fls)]
    return (torch.stack([r[0] for r in res[:n]]), torch.stack([r[1] for r in res[:n]]),
            torch.stack([r[0] for r in res[n:]]), torch.stack([r[1] for r in res[n:]]), gts)

def build_cache(items):
    """탐지기 고정 + Detect 직전 주입 -> 압축 P3·P4·P5, 원본 P3·P4는 항목마다 고정. CPU에 fp16으로 저장"""
    C = {}; ref_done = {}
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        m.eval()
        for j in range(0, len(items), 8):
            chunk = items[j:j + 8]; b = make_batch(chunk, flip=False)
            fc = L.neck_feats(m, b[0].to(DEV))
            fr = L.neck_feats(m, b[2].to(DEV))
            for q, it in enumerate(chunk):
                C[it] = [t[q].half().cpu() for t in fc]
                if (it[0], it[1]) not in ref_done: ref_done[(it[0], it[1])] = [t[q].half().cpu() for t in fr]
    return C, ref_done

def train_run(cfg):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    tr = [s for s in TRAIN_ALL if s != cfg["val_seq"]]
    ensure(tr + [cfg["val_seq"]], sorted(set(cfg["train_kbps"]) | set(cfg["eval_val_kbps"])))
    mu, sg, sgd = dct_stats(tr, cfg["train_kbps"])
    TB = torch.as_tensor(L.BANDS[cfg["tgt_band"]]); SGD = torch.as_tensor(sgd[TB.numpy()], dtype=torch.float32, device=DEV).view(1, -1, 1, 1)
    fcb = L.make_fcb(cfg, mu, sg).to(DEV); print("  params", fcb.n_params())
    frames = [(s, i) for s in tr for i in range(0, len(DATA[s]["gt"]), cfg["frame_step"])]
    steps_ep = max(1, len(frames) // cfg["batch"]); total = steps_ep * cfg["epochs"]; warm = steps_ep * cfg["warmup"]
    opt = torch.optim.AdamW(fcb.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    cache = refc = None
    if cfg["cache"]:
        key = (tuple(tr), cfg["frame_step"], tuple(cfg["train_kbps"]))
        if key not in CACHE:
            CACHE.clear(); gc.collect()
            t0 = time.time(); allit = [(s, i, k) for s, i in frames for k in cfg["train_kbps"]]
            CACHE[key] = build_cache(allit); print(f"  cache {len(CACHE[key][0])} items {time.time() - t0:.0f}s")
        cache, refc = CACHE[key]
    hist = []; t0 = time.time(); L.set_train_mode(m)
    for ep in range(cfg["epochs"]):
        order = rng.permutation(len(frames)); agg = {}
        for st in range(steps_ep):
            items = [(*frames[j], int(rng.choice(cfg["train_kbps"]))) for j in order[st * cfg["batch"]:(st + 1) * cfg["batch"]]]
            if cfg["cache"]:
                _, yc, _, yr, gts = make_batch(items, flip=False)                       # 캐시 모드: 반전 없음
                dt = torch.float16 if AMP else torch.float32
                fc = [torch.stack([cache[it][l] for it in items]).to(DEV, dt) for l in range(3)]
                fr = [torch.stack([refc[(it[0], it[1])][l] for it in items]).to(DEV, dt) for l in range(3)]
            else:
                rc, yc, rr, yr, gts = make_batch(items, cfg["flip"])
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    fc = L.neck_feats(m, rc.to(DEV)); fr = L.neck_feats(m, rr.to(DEV))
            yc, yr = yc.to(DEV), yr.to(DEV)
            with torch.no_grad(): tgt = ((dct(yr) - dct(yc))[:, TB] / SGD)[:, :, 1:-1]
            fcb.train(); L.set_train_mode(m)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                d3, d4, aux = L.run_fcb(fcb, yc, fc); p3, p4 = fc[0] + d3, fc[1] + d4
                preds = m.model[-1]([p3, p4, fc[2]])
            o = {k: (v.float() if torch.is_tensor(v) else [t.float() for t in v]) for k, v in preds["one2many"].items()}
            ld, _ = L.l_det(crit, {"one2many": o}, {k: v.to(DEV) for k, v in L.yolo_targets(gts).items()})
            p3, p4 = p3.float(), p4.float(); r3, r4 = fr[0].float(), fr[1].float()
            lg = L.rel_err(p3, r3)[0] + L.rel_err(p4, r4)[0]
            ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
            ls = L.rel_err(p3, r3, ms[0])[0] + L.rel_err(p4, r4, ms[1])[0]
            lf = ((aux.float()[:, :, 1:-1] - tgt) ** 2).mean() if aux is not None else torch.zeros((), device=DEV)
            lkc = lkb = lr = torch.zeros((), device=DEV)
            if cfg["lam_kd"] > 0:                                      # 교사: 같은 탐지기에 원본 feature
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    t_out = m.model[-1]([fr[0], fr[1], fr[2]])["one2many"]
                aw = L.anchor_small_weight(gts, L.CCTV + L.BALL).to(DEV) if cfg["kd_small"] else None
                lkc, lkb = L.kd_loss(o, t_out, aw)
            if cfg["lam_reg"] > 0:                                     # 보정량 크기 규제 (feature 에너지 대비)
                lr = (d3.float() ** 2).mean() / (fc[0].float() ** 2).mean() + (d4.float() ** 2).mean() / (fc[1].float() ** 2).mean()
            loss = (cfg["lam_det"] * ld + cfg["lam_feat"] * (lg + ls) + cfg["lam_freq"] * lf
                    + cfg["lam_kd"] * (lkc + lkb) + cfg["lam_reg"] * lr)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(fcb.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            for k, v in {"L_det": ld, "L_glob": lg, "L_small": ls, "L_freq": lf, "L_kdc": lkc, "L_kdb": lkb, "L_reg": lr, "loss": loss}.items(): agg[k] = agg.get(k, 0) + v.item() / steps_ep
        agg["epoch"] = ep; agg["min"] = round((time.time() - t0) / 60, 2); hist.append(agg)
        print(f"  ep{ep} " + " ".join(f"{k} {v:.4f}" for k, v in agg.items() if k.startswith("L") or k == "loss") + f" | {agg['min']} min")
    return fcb, hist, (mu, sg, sgd)

def eval_seq(s, kbs, fcb=None, step=1, group=None, keep_preds=False):
    """평가 시퀀스 하나: 버전(k)별 COCO AP. 'ref' = 무압축 원본"""
    n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); ids = list(range(0, n, step))
    G = L.coco_gt(gt); rows, P = [], {}
    for k in kbs:
        Y, U, V = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n) if k == "ref" else L.read_yuv(comp_path(s, k), n)
        dets = []
        for j in range(0, len(ids), 8):
            rgb, yp = map(torch.stack, zip(*POOL.map(lambda i: L.frame_tensors(Y[i], U[i], V[i]), ids[j:j + 8])))
            dets += L.detect(m, rgb.to(DEV), fcb, yp.to(DEV) if fcb is not None else None)
        pr = L.to_coco(dets, ids)
        for grp, cls in ([(group, L.CCTV if group == "cctv" else L.BALL)] if group else [("cctv", L.CCTV), ("ball", L.BALL)]):
            r = L.coco_ap(G, pr, cls, ids)
            if r: rows.append({"seq": s, "kbps": k, "group": grp, **r})
        if keep_preds: P[k] = np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in zip(ids, dets)]).astype(np.float32)
        del Y, U, V
    return rows, P

def eval_groups(fcb, kbs, tag, keep_preds=True):
    """평가셋: 그룹(CCTV: HighwayView+CampusView / ball: 3개) 합쳐서 AP. 프레임 예측은 bootstrap용 저장"""
    rows = []
    for grp in ["cctv", "ball"]:
        seqs = [s for s, g in EVAL.items() if g == grp]; cls = L.CCTV if grp == "cctv" else L.BALL
        for k in kbs:
            imgs, anns, preds, off = [], [], [], 0
            for s in seqs:
                n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); G = L.coco_gt(gt)
                Y, U, V = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n) if k == "ref" else L.read_yuv(comp_path(s, k), n)
                dets = []
                for j in range(0, n, 8):
                    rgb, yp = map(torch.stack, zip(*POOL.map(lambda i: L.frame_tensors(Y[i], U[i], V[i]), range(j, min(n, j + 8)))))
                    dets += L.detect(m, rgb.to(DEV), fcb, yp.to(DEV) if fcb is not None else None)
                imgs += [{**im, "id": im["id"] + off} for im in G["images"]]
                anns += [{**a, "id": a["id"] + off * 1000, "image_id": a["image_id"] + off} for a in G["annotations"]]
                preds += L.to_coco(dets, [i + off for i in range(n)])
                if keep_preds:
                    np.save(f"{OUT}/{tag}_preds_{s}_{k}.npy", np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in enumerate(dets)]).astype(np.float32))
                off += 10000; del Y, U, V
            r = L.coco_ap({"images": imgs, "annotations": anns, "categories": G["categories"]}, preds, cls)
            rows.append({"run": tag, "group": grp, "kbps": k, **(r or {})}); print("  ", tag, grp, k, r)
    return rows

def oracle_eval(kbs=(500, 250), alphas=(0.25, 0.5, 1.0), tag="oracle"):
    """정답 보정량 주입 상한: P_l + α·(P_ref,l − P_comp,l). 학습 없음.
    설정: base(α=0), a{α} (P3·P4), a1_small (작은 GT 박스 셀만, α=1), a1_P3, a1_P4"""
    cfgs = {"base": (0, 1, 1, False)} | {f"a{a}": (a, 1, 1, False) for a in alphas} | \
           {"a1_small": (1, 1, 1, True), "a1_P3": (1, 1, 0, False), "a1_P4": (1, 0, 1, False)}
    rows = []
    for grp in ["cctv", "ball"]:
        seqs = [s for s, g in EVAL.items() if g == grp]; cls = L.CCTV if grp == "cctv" else L.BALL
        for k in kbs:
            imgs, anns, preds, off = [], [], {c: [] for c in cfgs}, 0
            for s in seqs:
                n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); G = L.coco_gt(gt)
                C = L.read_yuv(comp_path(s, k), n); R = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n)
                dets = {c: [] for c in cfgs}
                for j in range(0, n, 8):
                    ids = list(range(j, min(n, j + 8)))
                    rc = torch.stack([r[0] for r in POOL.map(lambda i: L.frame_tensors(C[0][i], C[1][i], C[2][i]), ids)]).to(DEV)
                    rr = torch.stack([r[0] for r in POOL.map(lambda i: L.frame_tensors(R[0][i], R[1][i], R[2][i]), ids)]).to(DEV)
                    with torch.no_grad():
                        m.eval(); fc = L.neck_feats(m, rc); fr = L.neck_feats(m, rr)
                        ms = [x.to(DEV) for x in L.small_masks([gt[i] for i in ids], [(92, 160), (46, 80)])]
                        for c, (a, u3, u4, sm) in cfgs.items():
                            d3 = a * u3 * (fr[0] - fc[0]) * (ms[0] if sm else 1); d4 = a * u4 * (fr[1] - fc[1]) * (ms[1] if sm else 1)
                            dets[c] += L.detect_from_feats(m, [fc[0] + d3, fc[1] + d4, fc[2]])
                imgs += [{**im, "id": im["id"] + off} for im in G["images"]]
                anns += [{**a_, "id": a_["id"] + off * 1000, "image_id": a_["image_id"] + off} for a_ in G["annotations"]]
                for c in cfgs:
                    preds[c] += L.to_coco(dets[c], [i + off for i in range(n)])
                    np.save(f"{OUT}/{tag}_{c}_preds_{s}_{k}.npy", np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in enumerate(dets[c])]).astype(np.float32))
                off += 10000; del C, R
            for c in cfgs:
                r = L.coco_ap({"images": imgs, "annotations": anns, "categories": G["categories"]}, preds[c], cls)
                rows.append({"run": f"{tag}_{c}", "group": grp, "kbps": k, **(r or {})}); print("  ", tag, c, grp, k, r)
    pd.DataFrame(rows).to_csv(f"{OUT}/oracle_{STAMP}.csv", index=False)
    return rows

# ---- 실행 ----
ALL = []
for run in RUNS:
    cfg = {**DEFAULT, **run}; tag = f"{cfg['name']}"
    print(f"=== {tag} ===", {k: v for k, v in cfg.items() if v != DEFAULT.get(k)})
    if cfg["name"] == "baseline":
        fcb, hist = None, []
    elif cfg["ckpt"]:                                   # 저장된 FCB 가중치로 평가만
        ck = torch.load(cfg["ckpt"], map_location=DEV); c0 = ck["cfg"]
        fcb = L.make_fcb({**DEFAULT, **c0}, np.zeros(64), np.ones(64)).to(DEV)
        fcb.load_state_dict(ck["state"]); hist = ck.get("hist", [])
        ensure([cfg["val_seq"]], cfg["eval_val_kbps"])
    else:
        fcb, hist, st = train_run(cfg)
        torch.save({"cfg": cfg, "state": fcb.state_dict(), "hist": hist}, f"{OUT}/{tag}.pt")
    vrows, _ = eval_seq(cfg["val_seq"], cfg["eval_val_kbps"], fcb, group="cctv")
    for r in vrows: r.update(run=tag, split="val")
    ALL += vrows; print("  val", [(r["kbps"], r["APs"]) for r in vrows])
    if cfg["eval_full"]:
        er = eval_groups(fcb, cfg["eval_kbps"], tag)
        for r in er: r["split"] = "eval"
        ALL += er
    pd.DataFrame(ALL).to_csv(f"{OUT}/results_{STAMP}.csv", index=False)
    json.dump({"cfg": cfg, "hist": hist}, open(f"{OUT}/{tag}_log.json", "w"), default=str)
print(pd.DataFrame(ALL).to_string())


In [ ]:
import warnings, sys; warnings.filterwarnings('ignore')
sys.modules.pop('fcb_lib', None)   # 같은 세션에서 이전 버전 fcb_lib가 import돼 있으면 새로 읽게
SRC = open('/kaggle/working/fcb_train.py').read().split('# ---- 실행 ----')
exec(SRC[0])                       # 함수·설정·탐지기만 불러옴
assert comp_path('JobFair', 500) and comp_path('JobFair', 250), 'JobFair 압축 영상 없음'
print('JobFair frames', n_frames('JobFair'))

## Part A. 오라클 주입 (학습 없음, 약 15분)
압축 프레임의 P3·P4에 원본 프레임과의 차이를 그대로 더해서 평가: `P_l + α·(P_ref,l − P_comp,l)`. P5는 압축 그대로.

| 설정 | 내용 |
|---|---|
| base | α=0 (Baseline과 같아야 함) |
| a0.25 / a0.5 / a1.0 | P3·P4 전체에 α배 |
| a1_small | α=1, 작은 GT 박스(면적 < 32²)와 겹치는 셀만 |
| a1_P3 / a1_P4 | α=1, 한 레벨만 |

평가셋 5개 시퀀스, 500·250 kbps. 예측은 `oracle_{설정}_preds_{seq}_{k}.npy`로 저장.

In [ ]:
ORACLE = oracle_eval(kbs=(500, 250))
od = pd.DataFrame(ORACLE)
print(od.pivot_table(index='run', columns=['group', 'kbps'], values='APs').round(2).to_string())

In [ ]:
# 시퀀스별 APs (저장된 예측으로 계산)
rows = []
for s, grp in EVAL.items():
    n = n_frames(s); G = L.coco_gt(L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n)); cls = L.CCTV if grp == 'cctv' else L.BALL
    for k in (500, 250):
        for f in sorted(glob.glob(f"{OUT}/oracle_*_preds_{s}_{k}.npy")):
            c = os.path.basename(f)[len('oracle_'):-len(f'_preds_{s}_{k}.npy')]; a = np.load(f)
            dets = [a[a[:, 0] == i][:, 1:] for i in range(n)]
            r = L.coco_ap(G, L.to_coco(dets, list(range(n))), cls)
            rows.append({'seq': s, 'kbps': k, 'cfg': c, 'APs': None if r is None else r['APs']})
ps = pd.DataFrame(rows); ps.to_csv(f"{OUT}/oracle_per_seq_{STAMP}.csv", index=False)
print(ps.pivot_table(index='cfg', columns=['seq', 'kbps'], values='APs').round(2).to_string())

## Part B. 출력 증류 스크리닝 (약 30–40분)
교사 = 같은 고정 탐지기에 **원본 프레임** feature를 넣은 Detect 출력(one2many). 학생 = 압축 feature + FCB 보정.
- 분류: 교사 확률을 soft target으로 BCE, 앵커 가중 = max(교사 최대확률, 학생 최대확률)
- 박스: L1, 앵커 가중 = 교사 최대확률
- 규제: `lam_reg`·(‖ΔP3‖²/‖P3‖² + ‖ΔP4‖²/‖P4‖²)
- λ_det = λ_feat = λ_freq = 0 (GT 라벨 손실 없음)
- 검증 시퀀스: **JobFair** (평가셋·학습셋 어디에도 없음) → 학습은 6개 시퀀스 전부
- 메모리 때문에 frame_step 3 (나머지는 `SCREEN`과 같음: 캐시, 반전 없음, 500·250 kbps, 10 epoch)

| run | 설정 |
|---|---|
| K1 | FCB(DCT 입력) + 증류 |
| K2 | K1 + 작은 GT 박스(CCTV·ball 클래스) 안 앵커 5배 가중 |
| K3 | CondFCB `arch="feat"`(압축 P3·P4 feature 입력) + 증류 |

In [ ]:
KD = dict(lam_det=0, lam_feat=0, lam_freq=0, lam_kd=1.0, lam_reg=0.1, val_seq='JobFair', **{**SCREEN, 'frame_step': 3})
RUNS = [dict(name='K1_kd', **KD),
        dict(name='K2_kd_small', kd_small=True, **KD),
        dict(name='K3_kd_feat', arch='feat', **KD)]
exec('# ---- 실행 ----' + SRC[1])

In [ ]:
# 검증(JobFair) Baseline: 같은 표에 비교용
vb, _ = eval_seq('JobFair', [500, 250], None, group='cctv')
print(pd.DataFrame(vb)[['seq', 'kbps', 'AP', 'APs', 'APm', 'APl']].to_string())
pd.DataFrame(vb).to_csv(f"{OUT}/val_baseline_JobFair_{STAMP}.csv", index=False)

In [ ]:
!cd /kaggle/working && zip -qr fcb_runs.zip fcb_runs && ls -lh fcb_runs.zip